# Ollama Health Check Monitor

Monitoring cho Ollama service trên A100 Docker JupyterLab.

**Models:**
- qwen2.5:7b (LLM)
- nomic-embed-text (Embeddings)

**Ollama URL:** http://localhost:11434

**Cloudflare Tunnel:** https://grew-hypothesis-mothers-flooring.trycloudflare.com

In [ ]:
# Cell 1: Install dependencies
!pip install requests pandas matplotlib

In [ ]:
# Cell 2: Ollama health check functions
import requests
import time
from datetime import datetime
import json

OLLAMA_URL = 'http://localhost:11434'
EXPECTED_MODELS = ['qwen2.5:7b', 'nomic-embed-text']

def check_ollama_health():
    """Check if Ollama service is running and healthy"""
    try:
        response = requests.get(f'{OLLAMA_URL}/api/tags', timeout=5)
        
        if response.ok:
            data = response.json()
            models = [m['name'] for m in data.get('models', [])]
            
            # Check if expected models are loaded
            missing_models = [m for m in EXPECTED_MODELS if m not in models]
            
            return {
                'status': 'healthy',
                'models': models,
                'missing_models': missing_models,
                'timestamp': datetime.now().isoformat(),
                'response_time_ms': response.elapsed.total_seconds() * 1000
            }
        else:
            return {
                'status': 'unhealthy',
                'error': f'HTTP {response.status_code}',
                'timestamp': datetime.now().isoformat()
            }
    except requests.exceptions.Timeout:
        return {
            'status': 'timeout',
            'error': 'Request timeout (>5s)',
            'timestamp': datetime.now().isoformat()
        }
    except Exception as e:
        return {
            'status': 'down',
            'error': str(e),
            'timestamp': datetime.now().isoformat()
        }

def test_ollama_generation(model='qwen2.5:7b', prompt='Hello'):
    """Test Ollama generation endpoint"""
    try:
        start_time = time.time()
        response = requests.post(
            f'{OLLAMA_URL}/api/generate',
            json={'model': model, 'prompt': prompt, 'stream': False},
            timeout=30
        )
        elapsed = (time.time() - start_time) * 1000
        
        if response.ok:
            data = response.json()
            return {
                'status': 'success',
                'response_time_ms': elapsed,
                'response_length': len(data.get('response', '')),
                'timestamp': datetime.now().isoformat()
            }
        else:
            return {
                'status': 'failed',
                'error': f'HTTP {response.status_code}',
                'timestamp': datetime.now().isoformat()
            }
    except Exception as e:
        return {
            'status': 'error',
            'error': str(e),
            'timestamp': datetime.now().isoformat()
        }

def test_ollama_embedding(model='nomic-embed-text', text='test'):
    """Test Ollama embedding endpoint"""
    try:
        start_time = time.time()
        response = requests.post(
            f'{OLLAMA_URL}/api/embeddings',
            json={'model': model, 'prompt': text},
            timeout=10
        )
        elapsed = (time.time() - start_time) * 1000
        
        if response.ok:
            data = response.json()
            embedding = data.get('embedding', [])
            return {
                'status': 'success',
                'response_time_ms': elapsed,
                'embedding_dim': len(embedding),
                'timestamp': datetime.now().isoformat()
            }
        else:
            return {
                'status': 'failed',
                'error': f'HTTP {response.status_code}',
                'timestamp': datetime.now().isoformat()
            }
    except Exception as e:
        return {
            'status': 'error',
            'error': str(e),
            'timestamp': datetime.now().isoformat()
        }

print("✅ Ollama health check functions loaded")

In [ ]:
# Cell 3: Continuous health monitoring
import pandas as pd
from IPython.display import clear_output

health_history = []
max_history = 100

print("🚀 Starting Ollama health monitoring... (Press Interrupt to stop)\n")

try:
    while True:
        health = check_ollama_health()
        health_history.append(health)
        
        if len(health_history) > max_history:
            health_history.pop(0)
        
        clear_output(wait=True)
        
        print("🔍 Ollama Service Health Check")
        print("=" * 60)
        print(f"Timestamp: {health['timestamp']}")
        print(f"Status: {health['status'].upper()}")
        
        if health['status'] == 'healthy':
            print(f"✅ Service is healthy")
            print(f"Response Time: {health['response_time_ms']:.2f}ms")
            print(f"\nLoaded Models ({len(health['models'])})")
            for model in health['models']:
                status = '✅' if model in EXPECTED_MODELS else '⚠️'
                print(f"  {status} {model}")
            
            if health['missing_models']:
                print(f"\n⚠️ Missing Expected Models:")
                for model in health['missing_models']:
                    print(f"  ❌ {model}")
        else:
            print(f"❌ Service is {health['status']}")
            print(f"Error: {health.get('error', 'Unknown')}")
        
        print("=" * 60)
        
        # Calculate uptime
        if len(health_history) > 1:
            healthy_count = sum(1 for h in health_history if h['status'] == 'healthy')
            uptime_percent = (healthy_count / len(health_history)) * 100
            print(f"\n📊 Uptime (last {len(health_history)} checks): {uptime_percent:.1f}%")
        
        time.sleep(30)  # Check every 30 seconds
        
except KeyboardInterrupt:
    print("\n\n🛑 Monitoring stopped")
    
    if health_history:
        df = pd.DataFrame(health_history)
        filename = f"ollama_health_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        df.to_csv(filename, index=False)
        print(f"📁 Health history saved to: {filename}")

In [ ]:
# Cell 4: Quick health check (single snapshot)
print("🔍 Ollama Quick Health Check\n")

# Check service
health = check_ollama_health()
print(f"Service Status: {health['status'].upper()}")

if health['status'] == 'healthy':
    print(f"✅ Ollama is running")
    print(f"Response Time: {health['response_time_ms']:.2f}ms")
    print(f"\nModels: {', '.join(health['models'])}")
    
    # Test generation
    print("\n🧪 Testing generation endpoint...")
    gen_result = test_ollama_generation()
    if gen_result['status'] == 'success':
        print(f"✅ Generation working ({gen_result['response_time_ms']:.0f}ms)")
    else:
        print(f"❌ Generation failed: {gen_result.get('error')}")
    
    # Test embedding
    print("\n🧪 Testing embedding endpoint...")
    emb_result = test_ollama_embedding()
    if emb_result['status'] == 'success':
        print(f"✅ Embedding working ({emb_result['response_time_ms']:.0f}ms, dim={emb_result['embedding_dim']})")
    else:
        print(f"❌ Embedding failed: {emb_result.get('error')}")
else:
    print(f"❌ Ollama is {health['status']}")
    print(f"Error: {health.get('error')}")

In [ ]:
# Cell 5: Performance benchmark
print("⚡ Ollama Performance Benchmark\n")

# Benchmark generation
print("Testing qwen2.5:7b generation...")
gen_times = []
for i in range(5):
    result = test_ollama_generation(prompt=f"Test {i+1}")
    if result['status'] == 'success':
        gen_times.append(result['response_time_ms'])
        print(f"  Run {i+1}: {result['response_time_ms']:.0f}ms")

if gen_times:
    print(f"\nGeneration Stats:")
    print(f"  Average: {sum(gen_times)/len(gen_times):.0f}ms")
    print(f"  Min: {min(gen_times):.0f}ms")
    print(f"  Max: {max(gen_times):.0f}ms")

# Benchmark embedding
print("\nTesting nomic-embed-text embedding...")
emb_times = []
for i in range(10):
    result = test_ollama_embedding(text=f"Test document {i+1}")
    if result['status'] == 'success':
        emb_times.append(result['response_time_ms'])
        print(f"  Run {i+1}: {result['response_time_ms']:.0f}ms")

if emb_times:
    print(f"\nEmbedding Stats:")
    print(f"  Average: {sum(emb_times)/len(emb_times):.0f}ms")
    print(f"  Min: {min(emb_times):.0f}ms")
    print(f"  Max: {max(emb_times):.0f}ms")